In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField,
                                LongType, StringType, DoubleType)
import os, time
import os, time, glob, shutil, subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
spark = (SparkSession.builder
    .appName("Alibaba-MS-Load-Optimized")
 
    # ── Memory ──
    .config("spark.driver.memory",            "8g")
    .config("spark.executor.memory",          "20g")
    .config("spark.executor.memoryOverhead",  "4g")   # off-heap buffer
    .config("spark.driver.maxResultSize",     "4g")
 
    # ── Cores ──
    .config("spark.executor.cores",           "4")
 
    # ── Shuffle & parallelism ──
    # 200 partitions default is too low for 60 GB
    # Rule: ~128 MB per partition → 60 GB / 128 MB ≈ 480 partitions
    .config("spark.sql.shuffle.partitions",   "480")
    .config("spark.default.parallelism",      "96")   # 2 executors × 4 cores × 12
 
    # ── CSV reading ──
    .config("spark.sql.files.maxPartitionBytes", "134217728")  # 128 MB
    .config("spark.sql.files.openCostInBytes",   "4194304")    # 4 MB
 
    # ── Arrow (fast pandas conversion) ──
    .config("spark.sql.execution.arrow.pyspark.enabled",         "true")
    .config("spark.sql.execution.arrow.maxRecordsPerBatch",      "50000")
 
    # ── Parquet ──
    .config("spark.sql.parquet.compression.codec",   "snappy")
    .config("spark.sql.parquet.mergeSchema",          "false")  # faster reads
    .config("spark.sql.parquet.filterPushdown",       "true")
    .config("spark.sql.parquet.enableVectorizedReader","true")  # columnar scan
 
    # ── Adaptive Query Execution (Spark 3+) ──
    # Automatically coalesces partitions and optimizes joins at runtime
    .config("spark.sql.adaptive.enabled",                        "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled",     "true")
    .config("spark.sql.adaptive.skewJoin.enabled",               "true")
 
    # ── Spill to disk (safety net) ──
    .config("spark.local.dir",  "/tmp/spark-scratch")
    .config("spark.memory.fraction",          "0.7")
    .config("spark.memory.storageFraction",   "0.4")
 
    .getOrCreate())

26/04/28 10:50:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/28 10:50:35 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).


In [3]:
spark.sparkContext.setLogLevel("WARN")

In [4]:
print(f"Spark version: {spark.version}")
print(f"Executors: {spark.sparkContext.defaultParallelism}")

Spark version: 3.1.3
Executors: 96


In [93]:
DATA_ROOT   = "/home/s20426/S20426_Research_Microservices_Dataset/clusterdata/cluster-trace-microservices-v2021/data"
ARQUET_DIR = "data_parquet"
 
# [MODIFIED] added EXTRACT_DIR for MSResource and MSRTQps tar.gz extraction
#            MSCallGraph does not use this — its CSVs are already readable
EXTRACT_DIR = "data_extracted"
os.makedirs(EXTRACT_DIR, exist_ok=True)
# [END MODIFIED]
 
os.makedirs(PARQUET_DIR, exist_ok=True)

## Data Extraction 

In [86]:
def _extract_one(tb, out_dir):
    """Extract a single tar.gz using the system tar command."""
    csv_name = os.path.basename(tb).replace(".tar.gz", ".csv")
    out_file = os.path.join(out_dir, csv_name)
 
    # Skip if already extracted successfully
    if os.path.exists(out_file) and os.path.getsize(out_file) > 0:
        return f"  skip  {csv_name} (already exists)"
 
    result = subprocess.run(
        ["tar", "-xzf", tb, "-C", out_dir])
 
    if result.returncode != 0:
        return f"  ERROR {os.path.basename(tb)}: {result.stderr.strip()}"
 
    size_mb = os.path.getsize(out_file) / 1e6 if os.path.exists(out_file) else 0
    return f"  done  {csv_name}  ({size_mb:.0f} MB)"

In [87]:


def extract_parallel(table, max_workers=4):
    """
    Extract all tar.gz archives for a table in parallel.
    Output: EXTRACT_DIR/table/TableName_N.csv
    """
    out_dir  = os.path.join(EXTRACT_DIR, table)
    os.makedirs(out_dir, exist_ok=True)
    tarballs = sorted(glob.glob(f"{DATA_ROOT}/{table}/*.tar.gz"))
 
    if not tarballs:
        raise FileNotFoundError(
            f"No tar.gz found in {DATA_ROOT}/{table}/")
 
    print(f"\n  {table}: extracting {len(tarballs)} archives "
          f"with {max_workers} parallel workers ...")
    t0 = time.time()
 
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = {pool.submit(_extract_one, tb, out_dir): tb
                   for tb in tarballs}
        for future in as_completed(futures):
            print(future.result())
 
    csv_files = glob.glob(os.path.join(out_dir, "*.csv"))
    size_gb   = sum(os.path.getsize(f) for f in csv_files) / 1e9
    print(f"  ✓ {table}: {len(csv_files)} CSVs ready, "
          f"{size_gb:.1f} GB  ({(time.time()-t0)/60:.1f} min)")
 
 
print("\n── Extracting tar.gz (MSResource & MSRTQps only) ──")
extract_parallel("MSResource", max_workers=4)
extract_parallel("MSRTQps",    max_workers=4)


── Extracting tar.gz (MSResource & MSRTQps only) ──

  MSResource: extracting 12 archives with 4 parallel workers ...
  done  MSResource_0.csv  (2882 MB)
  done  MSResource_10.csv  (2889 MB)
  done  MSResource_11.csv  (2889 MB)
  done  MSResource_1.csv  (2885 MB)
  done  MSResource_2.csv  (2887 MB)
  done  MSResource_3.csv  (2897 MB)
  done  MSResource_4.csv  (2896 MB)
  done  MSResource_5.csv  (2895 MB)
  done  MSResource_6.csv  (2895 MB)
  done  MSResource_7.csv  (2894 MB)
  done  MSResource_8.csv  (2894 MB)
  done  MSResource_9.csv  (2890 MB)
  ✓ MSResource: 12 CSVs ready, 34.7 GB  (2.1 min)

  MSRTQps: extracting 25 archives with 4 parallel workers ...
  done  MSRTQps_1.csv  (2803 MB)
  done  MSRTQps_0.csv  (2784 MB)
  done  MSRTQps_10.csv  (2832 MB)
  done  MSRTQps_11.csv  (2832 MB)
  done  MSRTQps_12.csv  (2829 MB)
  done  MSRTQps_13.csv  (2829 MB)
  done  MSRTQps_15.csv  (2828 MB)
  done  MSRTQps_14.csv  (2830 MB)
  done  MSRTQps_18.csv  (2821 MB)
  done  MSRTQps_17.csv  (2828 

## MSRTQps 

In [88]:
def read_csv_fast(path, schema, name):
    t0 = time.time()
    df = (spark.read
          .option("header",        "true")
          .option("mode",          "DROPMALFORMED")  # skip bad rows silently
          .option("nullValue",     "")
          .option("nanValue",      "NaN")
          .option("compression",   "none")   # already extracted from tar.gz
          # Do NOT set inferSchema=true — we provide schema above
          .schema(schema)
          .csv(path))
    # Repartition to match parallelism budget
    df = df.repartition(480)
    print(f"  {name}: schema loaded in {time.time()-t0:.1f}s "
          f"(actual read happens on first action)")
    return df

In [108]:
from pyspark.sql.types import (StructType, StructField,
                                LongType, StringType, DoubleType,
                                IntegerType)

schema_rtqps_long = StructType([
    StructField("_c0",          IntegerType(), True),  # unnamed index
    StructField("timestamp",    LongType(),    True),
    StructField("msname",       StringType(),  True),
    StructField("msinstanceid", StringType(),  True),
    StructField("metric",       StringType(),  True),  # e.g. "providerRPC_MCR"
    StructField("value",        DoubleType(),  True),
])

In [114]:
# rtqps = read_csv_fast(
#     "/home/s20426/S20426_Research_Microservices_Dataset/clusterdata/cluster-trace-microservices-v2021/data/MSRTQps",
#     schema_rtqps,
#     "MSRTQps"
# )

# rtqps_csv_files = sorted(
#     glob.glob(os.path.join(EXTRACT_DIR, "MSRTQps")))
# rtqps = read_csv_fast(rtqps_csv_files, schema_rtqps, "MSRTQps")

rtqps_long = (spark.read
    .option("header",    "true")
    .option("mode",      "DROPMALFORMED")
    .option("nullValue", "")
    .schema(schema_rtqps_long)
    .csv("data_extracted/MSRTQps"))
 
# Check all distinct metric names before pivoting
print("\nDistinct metric values in MSRTQps:")
rtqps_long.select("metric").distinct().show(30, truncate=False)
 
# [FIXED] Aggregate to service level first (mean across instances),
#         then pivot metric column → wide format
rtqps = (rtqps_long
    .drop("_c0")
    # aggregate instances → service level
    .groupBy("timestamp", "msname", "metric")
    .agg(F.mean("value").alias("value"))
    # pivot: one column per metric
    .groupBy("timestamp", "msname")
    .pivot("metric")          # creates one col per distinct metric value
    .agg(F.first("value"))    # one value per (timestamp, msname, metric)
    .withColumn("t_idx", (F.col("timestamp") / 30000).cast("int")))
 
print("\nMSRTQps schema after pivot:")
rtqps.printSchema()
rtqps.show(3, truncate=True)


Distinct metric values in MSRTQps:
+---------------+
|metric         |
+---------------+
|consumerMQ_MCR |
|providerRPC_MCR|
|HTTP_RT        |
|consumerMQ_RT  |
|providerRPC_RT |
|consumerRPC_RT |
|HTTP_MCR       |
|consumerRPC_MCR|
+---------------+


MSRTQps schema after pivot:
root
 |-- timestamp: long (nullable = true)
 |-- msname: string (nullable = true)
 |-- HTTP_MCR: double (nullable = true)
 |-- HTTP_RT: double (nullable = true)
 |-- consumerMQ_MCR: double (nullable = true)
 |-- consumerMQ_RT: double (nullable = true)
 |-- consumerRPC_MCR: double (nullable = true)
 |-- consumerRPC_RT: double (nullable = true)
 |-- providerRPC_MCR: double (nullable = true)
 |-- providerRPC_RT: double (nullable = true)
 |-- t_idx: integer (nullable = true)

+---------+--------------------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+-----+
|timestamp|              msname|          HTTP_MCR|

In [115]:
def write_parquet(df, out_path, name):
    if os.path.exists(out_path):
        shutil.rmtree(out_path)
 
    t0 = time.time()
    print(f"\n── Writing {name} → Parquet ──")
 
    sample = df.limit(3).collect()
    if not sample:
        raise ValueError(f"{name}: DataFrame still empty")
    print(f"  Sample: {sample[0].asDict()}")
 
    (df.repartition(480)
       .write
       .mode("overwrite")
       .option("compression", "snappy")
       .parquet(out_path))
 
    elapsed  = time.time() - t0
    pq_files = glob.glob(os.path.join(out_path, "*.parquet"))
    pq_size  = sum(os.path.getsize(f) for f in pq_files) / 1e9
    print(f"  ✓ {len(pq_files)} files, {pq_size:.1f} GB "
          f"({elapsed/60:.1f} min)")
 
# write_parquet(resource, f"{PARQUET_DIR}/resource", "MSResource")
write_parquet(rtqps,    f"{PARQUET_DIR}/rtqps",    "MSRTQps")


── Writing MSRTQps → Parquet ──
  Sample: {'timestamp': 1080000, 'msname': 'fdfb8ee968d1aafdfe41941f991ae596375dcf18739349968a4fefa325e45198', 'HTTP_MCR': 7.260484622553588, 'HTTP_RT': 7.260484622553588, 'consumerMQ_MCR': 29.738001988125568, 'consumerMQ_RT': 29.738001988125568, 'consumerRPC_MCR': 25.196498599439774, 'consumerRPC_RT': 16.24133460844062, 'providerRPC_MCR': 28.56988795518207, 'providerRPC_RT': 18.043335318816766, 't_idx': 36}
  ✓ 480 files, 0.1 GB (2.4 min)


In [96]:
# rtqps = rtqps.withColumn(
#     "t_idx",    (F.col("timestamp") / 30000).cast("int")
# ).withColumn(
#     "t_bucket", ((F.col("timestamp") / 30000) / BUCKET_SIZE)
#                 .cast("int").cast("string")
# )

# rtqps = rtqps.withColumn(
#     "t_idx", (F.col("timestamp") / 30000).cast("int"))

In [97]:
# def write_parquet(df, out_path, partition_col=None, name=""):
#     t0 = time.time()
#     print(f"\n── Writing {name} → Parquet ──")
#     writer = df.write.mode("overwrite").format("parquet")
#     if partition_col:
#         writer = writer.partitionBy(partition_col)
#     writer.save(out_path)
#     elapsed = time.time() - t0
#     print(f"   Done in {elapsed/60:.1f} min  →  {out_path}")
# def write_parquet(df, out_path, name):
#     """Write DataFrame to flat Parquet (no partitioning)."""
#     if os.path.exists(out_path):
#         shutil.rmtree(out_path)
 
#     t0 = time.time()
#     print(f"\n── Writing {name} → Parquet ──")
 
#     # Spot-check before writing
#     sample = df.limit(3).collect()
#     if not sample:
#         raise ValueError(f"{name}: DataFrame is empty — check CSV paths")
#     print(f"  Sample: {sample[0].asDict()}")
 
#     (df.write
#        .mode("overwrite")
#        .option("compression", "snappy")
#        .parquet(out_path))
 
#     elapsed = time.time() - t0
#     print(f"  Done in {elapsed/60:.1f} min  →  {out_path}")
 
# # os.makedirs("./data", exist_ok=True)

In [123]:
# write_parquet(rtqps,
#               f"{PARQUET_DIR}/rtqps",
#               "MSRTQps")

## Resource Graph

In [119]:
schema_resource = StructType([
    StructField("_c0",                    IntegerType(), True),  # unnamed index
    StructField("msname",                 StringType(),  True),
    StructField("msinstanceid",           StringType(),  True),
    StructField("nodeid",                 StringType(),  True),
    StructField("instance_cpu_usage",     DoubleType(),  True),  # actual name
    StructField("instance_memory_usage",  DoubleType(),  True),  # actual name
    StructField("timestamp",              LongType(),    True),  # moved to end
])

In [120]:


# # resource  = read_csv_fast("/home/s20426/S20426_Research_Microservices_Dataset/clusterdata/cluster-trace-microservices-v2021/data/MSResource",
# #                            schema_resource,  "MSResource")


# resource_csv_files = sorted(
#     glob.glob(os.path.join(EXTRACT_DIR, "MSResource", "*.csv")))
# resource = read_csv_fast(resource_csv_files, schema_resource, "MSResource")

resource_files="data_extracted/MSResource"
resource_raw = (spark.read
    .option("header",    "true")
    .option("mode",      "DROPMALFORMED")
    .option("nullValue", "")
    .schema(schema_resource)
    .csv(resource_files))

In [121]:
# resource = resource.withColumn(
#     "t_idx",    (F.col("timestamp") / 30000).cast("int")
# ).withColumn(
#     "t_bucket", ((F.col("timestamp") / 30000) / BUCKET_SIZE)
#                 .cast("int").cast("string")
# )
# resource = resource.withColumn(
#     "t_idx", (F.col("timestamp") / 30000).cast("int"))
resource = (resource_raw
    .drop("_c0")
    .withColumnRenamed("instance_cpu_usage",    "cpu_utilization")
    .withColumnRenamed("instance_memory_usage",  "memory_utilization")
    .withColumn("t_idx", (F.col("timestamp") / 30000).cast("int")))
 
print("\nMSResource schema after fix:")
resource.printSchema()
resource.show(3, truncate=True)


MSResource schema after fix:
root
 |-- msname: string (nullable = true)
 |-- msinstanceid: string (nullable = true)
 |-- nodeid: string (nullable = true)
 |-- cpu_utilization: double (nullable = true)
 |-- memory_utilization: double (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- t_idx: integer (nullable = true)

+--------------------+--------------------+--------------------+-------------------+------------------+---------+-----+
|              msname|        msinstanceid|              nodeid|    cpu_utilization|memory_utilization|timestamp|t_idx|
+--------------------+--------------------+--------------------+-------------------+------------------+---------+-----+
|e969a0c7f24365fb7...|2d64adf48bfaa8e81...|85ad19cedb18370e1...| 0.1549999999968956| 0.676112174987793|        0|    0|
|9ffca37e87db8609a...|07dac0b810a2192b2...|5c23656eb3f2a97c0...|0.21154166666480403|0.6676225662231445|        0|    0|
|94dce029795f93575...|2b8cd6196fb98e11b...|93a0b91487e346215...|0.1691

In [122]:
write_parquet(resource,
              f"{PARQUET_DIR}/resource",
              "MSResource")


── Writing MSResource → Parquet ──
  Sample: {'msname': 'e969a0c7f24365fb761f28067720558309317afce8630dfbc12a2f8644dee2b6', 'msinstanceid': '2d64adf48bfaa8e81b72c6ec8e691a90d72237c944c4de775000e35e2d6523d4', 'nodeid': '85ad19cedb18370e11dd3f2f5ecde656b171464734e31b1de74af98a2bc30f87', 'cpu_utilization': 0.1549999999968956, 'memory_utilization': 0.676112174987793, 'timestamp': 0, 't_idx': 0}
  ✓ 480 files, 11.6 GB (1.9 min)


## Call Graph

In [5]:
import tarfile, glob, os, shutil, subprocess, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyspark.sql import functions as F
from pyspark.sql.types import *

DATA_ROOT   = "/home/s20426/S20426_Research_Microservices_Dataset/clusterdata/cluster-trace-microservices-v2021/data"
EXTRACT_DIR = "/home/s20426/S20426_Research_Microservices_Dataset/data_extracted"
PARQUET_DIR = "data_parquet"

# ── Corrected schema (9 actual columns) ──────────────────────────
# header: ,traceid,timestamp,rpcid,um,rpctype,dm,interface,rt
schema_callgraph = StructType([
    StructField("_c0",       IntegerType(), True),  # unnamed index
    StructField("traceid",   StringType(),  True),  # trace ID
    StructField("timestamp", LongType(),    True),
    StructField("rpcid",     StringType(),  True),  # call chain position
    StructField("um",        StringType(),  True),  # upstream (callee)
    StructField("rpctype",   StringType(),  True),  # mc/rpc/http/mq
    StructField("dm",        StringType(),  True),  # downstream (caller)
    StructField("interface", StringType(),  True),  # API endpoint
    StructField("rt",        DoubleType(),  True),  # response time
])

In [6]:


# ── Step 1: Extract tar.gz in parallel ───────────────────────────
def extract_one(tb, out_dir):
    csv_name = os.path.basename(tb).replace(".tar.gz", ".csv")
    out_file = os.path.join(out_dir, csv_name)
    if os.path.exists(out_file) and os.path.getsize(out_file) > 0:
        return f"  skip  {csv_name}"
    result = subprocess.run(["tar", "-xzf", tb, "-C", out_dir],
                            capture_output=True, text=True)
    if result.returncode != 0:
        return f"  ERROR {csv_name}: {result.stderr.strip()}"
    size_mb = os.path.getsize(out_file)/1e6 if os.path.exists(out_file) else 0
    return f"  done  {csv_name} ({size_mb:.0f} MB)"

out_dir  = os.path.join(EXTRACT_DIR, "MSCallGraph")
os.makedirs(out_dir, exist_ok=True)
tarballs = sorted(glob.glob(f"{DATA_ROOT}/MSCallGraph/*.tar.gz"))
print(f"Extracting {len(tarballs)} MSCallGraph archives ...")

with ThreadPoolExecutor(max_workers=4) as pool:
    futures = {pool.submit(extract_one, tb, out_dir): tb
               for tb in tarballs}
    for f in as_completed(futures):
        print(f.result())

csv_files = sorted(glob.glob(f"{out_dir}/*.csv"))
size_gb   = sum(os.path.getsize(f) for f in csv_files) / 1e9
print(f"\n✓ {len(csv_files)} CSVs, {size_gb:.1f} GB")

# ── Step 2: Read + write Parquet ─────────────────────────────────
shutil.rmtree(f"{PARQUET_DIR}/callgraph", ignore_errors=True)

df = (spark.read
      .option("header",    "true")
      .option("mode",      "DROPMALFORMED")
      .option("nullValue", "")
      .schema(schema_callgraph)
      .csv(csv_files)
      .drop("_c0")
      # Rename to standard names (uppercase DM/UM as before)
      .withColumnRenamed("dm", "DM")
      .withColumnRenamed("um", "UM")
      .withColumn("t_idx", (F.col("timestamp") / 30000).cast("int"))
      # Deduplicate on DM+UM+timestamp (keep all calls, not just unique pairs)
      .dropDuplicates(["DM", "UM", "timestamp"]))

# Spot check
print("\nSample rows:")
df.show(3, truncate=True)
print("Columns:", df.columns)

sample = df.limit(3).collect()
if not sample:
    raise ValueError("Empty DataFrame — check schema vs actual headers")

(df.repartition(16)
   .write
   .mode("overwrite")
   .option("compression", "snappy")
   .parquet(f"{PARQUET_DIR}/callgraph"))

print(f"\n✓ Callgraph written with {len(df.columns)} columns:")
print(f"  {df.columns}")

# ── Verify ────────────────────────────────────────────────────────
result = spark.read.parquet(f"{PARQUET_DIR}/callgraph")
print(f"\nFinal columns : {result.columns}")
print(f"Row count     : {result.count():,}")
result.show(3, truncate=True)

Extracting 145 MSCallGraph archives ...
  skip  MSCallGraph_1.csv
  skip  MSCallGraph_10.csv
  skip  MSCallGraph_101.csv
  skip  MSCallGraph_0.csv
  skip  MSCallGraph_103.csv
  skip  MSCallGraph_105.csv
  skip  MSCallGraph_100.csv
  skip  MSCallGraph_107.csv
  skip  MSCallGraph_106.csv
  skip  MSCallGraph_109.csv
  skip  MSCallGraph_102.csv
  skip  MSCallGraph_11.csv
  skip  MSCallGraph_104.csv
  skip  MSCallGraph_108.csv
  skip  MSCallGraph_111.csv
  skip  MSCallGraph_114.csv
  skip  MSCallGraph_113.csv
  skip  MSCallGraph_116.csv
  skip  MSCallGraph_115.csv
  skip  MSCallGraph_117.csv
  skip  MSCallGraph_118.csv
  skip  MSCallGraph_12.csv
  skip  MSCallGraph_120.csv
  skip  MSCallGraph_121.csv
  skip  MSCallGraph_119.csv
  skip  MSCallGraph_110.csv
  skip  MSCallGraph_122.csv
  skip  MSCallGraph_124.csv
  skip  MSCallGraph_125.csv
  skip  MSCallGraph_127.csv
  skip  MSCallGraph_123.csv
  skip  MSCallGraph_112.csv
  skip  MSCallGraph_128.csv
  skip  MSCallGraph_130.csv
  skip  MSCallG


Sample rows:


+--------------------+---------+-------------------+---+-----------+---+--------------------+-----+-----+
|             traceid|timestamp|              rpcid| UM|    rpctype| DM|           interface|   rt|t_idx|
+--------------------+---------+-------------------+---+-----------+---+--------------------+-----+-----+
|0b520727159192361...|    10932|0.1.1.2.1.1.8.5.2.1|(?)|        rpc|(?)|19bbbedd43d8c56ce...|  3.0|    0|
|0b5106ca159192362...|    20528|      0.1.1.2.7.1.0|(?)|userDefined|(?)|                null|278.0|    0|
|0b520627159192362...|    23932|   0.1.3.28.1.5.1.0|(?)|userDefined|(?)|                null|  3.0|    0|
+--------------------+---------+-------------------+---+-----------+---+--------------------+-----+-----+
only showing top 3 rows

Columns: ['traceid', 'timestamp', 'rpcid', 'UM', 'rpctype', 'DM', 'interface', 'rt', 't_idx']



✓ Callgraph written with 9 columns:
  ['traceid', 'timestamp', 'rpcid', 'UM', 'rpctype', 'DM', 'interface', 'rt', 't_idx']

Final columns : ['traceid', 'timestamp', 'rpcid', 'UM', 'rpctype', 'DM', 'interface', 'rt', 't_idx']
Row count     : 506,254,818
+--------------------+---------+------------+--------------------+-------+--------------------+--------------------+------+-----+
|             traceid|timestamp|       rpcid|                  UM|rpctype|                  DM|           interface|    rt|t_idx|
+--------------------+---------+------------+--------------------+-------+--------------------+--------------------+------+-----+
|0b520598159192978...|  6189524|0.1.1.2.1.40|35114acfb54c54fb9...|     mc|431a07b20e43caa33...|                null|   0.0|  206|
|0b144db9159193516...| 11567332|       0.1.1|                 (?)|   http|95a6f7f8345e2eca3...|b5d09f361bd1ea282...|-359.0|  385|
|0b13393a159192799...|  4391689|0.1.1.2.14.1|3cab0a98767379fbc...|    rpc|75e56c8fbb9336eb4...|d

### Reads parqet data

In [8]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

PARQUET_DIR = "data_parquet"

def load_resource(t_start=None, t_end=None):
    df = spark.read.parquet(f"{PARQUET_DIR}/resource")
    if t_start is not None: df = df.filter(F.col("t_idx") >= t_start)
    if t_end   is not None: df = df.filter(F.col("t_idx") <  t_end)
    return df

def load_rtqps(t_start=None, t_end=None):
    df = spark.read.parquet(f"{PARQUET_DIR}/rtqps")
    if t_start is not None: df = df.filter(F.col("t_idx") >= t_start)
    if t_end   is not None: df = df.filter(F.col("t_idx") <  t_end)
    return df

def load_callgraph():
    return spark.read.parquet(f"{PARQUET_DIR}/callgraph")

# Check row counts + schemas
for label, fn in [
    ("MSResource  (t 0–100)", lambda: load_resource(0, 100)),
    ("MSRTQps     (t 0–100)", lambda: load_rtqps(0, 100)),
    ("MSCallGraph (full)",     load_callgraph),
]:
    n = fn().count()
    print(f"  {'✓' if n > 0 else '✗'} {label}: {n:,} rows")

# Check column names — critical before EDA
print("\nMSResource columns:  ", load_resource(0,1).columns)
print("MSRTQps columns:     ", load_rtqps(0,1).columns)
print("MSCallGraph columns: ", load_callgraph().columns)

  ✓ MSResource  (t 0–100): 9,636,794 rows
  ✓ MSRTQps     (t 0–100): 64,987 rows
  ✓ MSCallGraph (full): 506,254,818 rows

MSResource columns:   ['msname', 'msinstanceid', 'nodeid', 'cpu_utilization', 'memory_utilization', 'timestamp', 't_idx']
MSRTQps columns:      ['timestamp', 'msname', 'HTTP_MCR', 'HTTP_RT', 'consumerMQ_MCR', 'consumerMQ_RT', 'consumerRPC_MCR', 'consumerRPC_RT', 'providerRPC_MCR', 'providerRPC_RT', 't_idx']
MSCallGraph columns:  ['traceid', 'timestamp', 'rpcid', 'UM', 'rpctype', 'DM', 'interface', 'rt', 't_idx']


isuru
